In [ ]:
1. Set up an environment for integration testing and write test cases that
validate the interaction between different components of a system?

Ans.

Step 1: Set Up the Environment (Dependencies)
To set up your environment, you need a build tool like Maven. Here is the exact pom.xml configuration you need to pull in the Spring Boot web framework, database tools, and testing libraries.



```
<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-web</artifactId>
</dependency>
<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-data-jpa</artifactId>
</dependency>

<dependency>
    <groupId>com.h2database</groupId>
    <artifactId>h2</artifactId>
    <scope>runtime</scope>
</dependency>

<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-test</artifactId>
    <scope>test</scope>
</dependency>
```
Step 2: The Components We Are Testing
To write an integration test, we need interacting components. Imagine a simple system that saves a user to a database.

The Entity: Represents the database table.

The Repository: Handles database queries.

The Service: Handles business logic.

The Controller: Exposes the HTTP endpoint.

Here is the condensed application code for context:



```
// 1. Entity
@Entity
public class User {
    @Id @GeneratedValue private Long id;
    private String username;
    // Getters and Setters omitted for brevity
}

// 2. Repository
public interface UserRepository extends JpaRepository<User, Long> {
    Optional<User> findByUsername(String username);
}

// 3. Service
@Service
public class UserService {
    @Autowired private UserRepository repository;
    
    public User createUser(User user) {
        return repository.save(user);
    }
}

// 4. Controller
@RestController
@RequestMapping("/api/users")
public class UserController {
    @Autowired private UserService service;

    @PostMapping
    public ResponseEntity<User> register(@RequestBody User user) {
        return new ResponseEntity<>(service.createUser(user), HttpStatus.CREATED);
    }
}
```

Step 3: Write the Integration TestNow, let's write the test case. We will use @SpringBootTest to load the entire application context (wiring all the components together) and MockMvc to simulate HTTP requests.This test validates the complete interaction: HTTP Client $\rightarrow$ Controller $\rightarrow$ Service $\rightarrow$ Repository $\rightarrow$ H2 Database.



```
import org.junit.jupiter.api.Test;
import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.boot.test.autoconfigure.web.servlet.AutoConfigureMockMvc;
import org.springframework.boot.test.context.SpringBootTest;
import org.springframework.http.MediaType;
import org.springframework.test.web.servlet.MockMvc;
import static org.springframework.test.web.servlet.request.MockMvcRequestBuilders.post;
import static org.springframework.test.web.servlet.result.MockMvcResultMatchers.status;
import static org.springframework.test.web.servlet.result.MockMvcResultMatchers.jsonPath;
import static org.junit.jupiter.api.Assertions.assertNotNull;

@SpringBootTest
@AutoConfigureMockMvc
public class UserIntegrationTest {

    @Autowired
    private MockMvc mockMvc; // Simulates HTTP requests

    @Autowired
    private UserRepository userRepository; // To verify data actually hit the DB

    @Test
    public void testUserRegistrationFlow_StoresInDatabase() throws Exception {
        // 1. Arrange: Prepare the JSON payload for the HTTP request
        String userJson = "{\"username\": \"integration_test_user\"}";

        // 2. Act: Simulate a POST request to the Controller
        mockMvc.perform(post("/api/users")
                .contentType(MediaType.APPLICATION_JSON)
                .content(userJson))
                
        // 3. Assert (Web Layer): Verify the Controller returns a 201 Created status
                .andExpect(status().isCreated())
                .andExpect(jsonPath("$.username").value("integration_test_user"));

        // 4. Assert (Database Layer): Verify the Service and Repository actually saved it
        User savedUser = userRepository.findByUsername("integration_test_user").orElse(null);
        
        assertNotNull(savedUser, "User should have been saved to the database");
    }
}
```

Why this is a good Integration Test:
No Mocks for Core Logic: Unlike a unit test where we would "fake" the database connection using Mockito, this test uses a real (though in-memory) database.

Cross-Boundary Validation: It proves that the JSON deserialization in the Controller, the saving logic in the Service, and the SQL generation in the Repository all work flawlessly together.